# Exercise 4: LLM API Calls (30 pts)

## Name: SIDNEY DARANGWA

This exercise must be sumbitted to Canvas by the end of class.

Remember to:
- **Share** the file with: jaredws@udel.edu & bonyy@udel.edu
- Export and submit the PDF version to Canvas
- Comment the Shared link of this Notebook to your `PDF` submission.

In [ ]:
# step 1: Mount the drive first
from google.colab import drive
drive.mount("/content/drive/")

# step 3: Add the path to your course folder
import os
os.chdir('/content/drive/MyDrive/MISY436')

## Install Langchain

In [ ]:
!pip install -qU "langchain[openai]"
!pip install -U langchain-community
!pip install pypdf

## Input Lerner Gemini API Keys

Copy and enter your API key when prompted below:

In [ ]:
import os
import getpass

from langchain.chat_models import init_chat_model

In [ ]:
# Enter your openai api key
# NEVER show your API key in the code!
os.environ["OPENAI_API_KEY"] = getpass.getpass(
    prompt="Enter your API key: "
)

## Play Around

Let's use the API to get a joke.

In [ ]:
## You don't need to change any of thise
## the LLM object below is important
## This set-up gives us access to UD's Enterprise License Gemini
## It is secure and data-private, following UD's Enterprise License
# initialize a model
llm = init_chat_model("gemini/gemini-3.1-flash-lite-preview",
                      base_url="https://lmgateway.lernercollegeit.org",
                      model_provider="openai")

# prepare the messages
# it's okay if you use "role: user"
messages = [
    {'role': 'system', 'content': 'Please tell me a joke'}
]

# "invoke" the model with your messages
response = llm.invoke(messages)

Let's print out the response!

In [ ]:
# print the response content (which is only a part of the response)
response.content

In Langchain, the response is of the `AIMessage` type, and it contains a lot of useful information. Some notable are:

- `complete_tokens`: the number of generated tokens (output)
- `prompt_tokens`: the number of prompt tokens (input)
- `total_tokens`: output and input tokens
- `model_name`: the model name

Let's print out the complete response!

In [ ]:
# print out the complete response
response

Let's caculate how much did our API call [cost](https://openai.com/api/pricing/).

As of 4/27/2026, the cost of Gemini 3.1 Flass Lite is:

- Input: $0.25 / 1M tokens

- Output: $1.50 / 1M tokens

> WARNING: the cost can easily build up for real world tasks. Always monitor your usage!

In [ ]:
## What's the cost of our response above?



## Classification


Task: We want to classify each of the following sentence into "positive", "negative", and "netural".

Let's use LLM to do this!

In [ ]:
# the sentences to be classified
sentences = [
    "The presentation was insightful and really well-delivered.",
    "Unfortunately, the project fell short of expectations.",
    "The meeting is scheduled for 3 PM tomorrow.",
    "Everyone appreciated your hard work and dedication.",
    "The software kept crashing, which was frustrating."
]

# initialize a model
llm = init_chat_model("gemini/gemini-3.1-flash-lite-preview",
                      base_url="https://lmgateway.lernercollegeit.org",
                      model_provider="openai")

# a placeholder for the results
responses = []

# for each sentence, call the api
for sentence in sentences:

    # construct the message for each sentence
    messages = [
        {'role': 'system', 'content': 'Please classify the following sentence into "positive", "negative", and "netural".'},
        {'role': 'user', 'content': sentence}
    ]

    # call api
    response = llm.invoke(messages)

    # save the response
    responses.append(response)

Let's print out the result:

In [ ]:
for response in responses:
    print(response.content)

Although the results are correct, the formatting is inconsistent — some responses include explanations while others don’t; some labels are capitalized (e.g., Negative), while others are lowercase (e.g., positive).

The inconsistent output formatting could create significant challenges for downstream processing.

Let’s update our system prompt to instruct GPT to format outputs consistently and cleanly.

In [ ]:
# the sentences to be classified
sentences = [
    "The presentation was insightful and really well-delivered.",
    "Unfortunately, the project fell short of expectations.",
    "The meeting is scheduled for 3 PM tomorrow.",
    "Everyone appreciated your hard work and dedication.",
    "The software kept crashing, which was frustrating."
]

# initialize a model
llm = init_chat_model("gemini/gemini-3.1-flash-lite-preview",
                      base_url="https://lmgateway.lernercollegeit.org",
                      model_provider="openai")


# for each sentence, call the api
for sentence in sentences:

    # construct the message for each sentence
    messages = [
        {'role': 'system', 'content': "Please classify the following sentence into positive, negative, and netural. Don't explain. Just return the result."},
        {'role': 'user', 'content': sentence}
    ]

    # call api
    response = llm.invoke(messages)

    # save the response
    print(response.content)

In the example above, we each input message consists of one "system message" and one "user message." We can combine both into one "user message."

In [ ]:
# the sentences to be classified
sentences = [
    "The presentation was insightful and really well-delivered.",
    "Unfortunately, the project fell short of expectations.",
    "The meeting is scheduled for 3 PM tomorrow.",
    "Everyone appreciated your hard work and dedication.",
    "The software kept crashing, which was frustrating."
]

# initialize a model
llm = init_chat_model("gemini/gemini-3.1-flash-lite-preview",
                      base_url="https://lmgateway.lernercollegeit.org",
                      model_provider="openai")


# for each sentence, call the api
for sentence in sentences:

    # construct the message for each sentence
    messages = [
        {'role': 'user', 'content': f"Please classify sentence \"{sentence}\" into positive, negative, and netural. Don't explain. Just return the result."},
    ]

    # call api
    response = llm.invoke(messages)

    # save the response
    print(response.content)

As you can see, the results are acceptable, but the output format is inconsistent—some labels are in lowercase, while others are in uppercase.

In practice, it’s best to include instructions (including formatting instructions) in the system message (which remains consistent across different invoke calls), and provide the data in the user message.

## TODO: Please generate five jokes about the following topics

```python
topics = ['cat', 'mouse', 'pineapple', 'Microsoft', 'Nvidia']
```

Requirement:
- You must use a `for` loop.
- Make sure the joke includes a punch line (or a twist)
- (Optional): Create two versions of solutions. In version 1, the input only contains one "user message" for each topic. In version 2, the input contains one "system message" and one "user message".

Hint:
- Use `f-string` when you construct the messages.

## Refine your CV

**Backgroud**

FintechX has a new job post below:

**Task**

You want to leverage a large language model (LLM) to refine your CV, tailoring it more effectively to the job description.

In [ ]:
# first, create the job description

job_description = """
    Job Title: Data Scientist – Financial AI

    Company: FinTechX

    Location: New York, NY (Hybrid)

    Job Description:
    We are seeking a data scientist with expertise in AI/ML and a strong background in finance.
    The ideal candidate has experience working with large language models (LLMs), time-series data,
    and algorithmic trading strategies.

    Responsibilities include:
    - Building and fine-tuning LLM-based models for financial use cases.
    - Collaborating with engineering and product teams to deliver scalable ML solutions.
    - Communicating insights to stakeholders through data storytelling and dashboards.

    Requirements:
    - MS/PhD in Computer Science, Statistics, Finance, or related fields.
    - Experience with LLMs (e.g., GPT, LLaMA), financial modeling, and Python (Polars preferred).
    - Familiarity with LangChain or similar frameworks a plus.
"""

First, please upload your CV (in a PDF) to Colab. If you haven't prepared your CV, please use this [link](https://drive.google.com/file/d/1TaeT9bLL9tKH2_jpL34cIXObVd6nyKXh/view?usp=sharing) to download an example CV.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

cv_path = "Data/MIS Resume.pdf"
loader = PyPDFLoader(cv_path)
cv_pages = loader.load_and_split()
cv_text = "\n".join([page.page_content for page in cv_pages])

print(cv_text)

In [ ]:
# initialize a model
llm = init_chat_model("gemini/gemini-3.1-flash-lite-preview",
                      base_url="https://lmgateway.lernercollegeit.org",
                      model_provider="openai")



# construct the messages
messages = [
    {'role': 'system', 'content': 'You are an expert resume consultant. Based on a given job description, help improve the CV to maximize alignment.'},
    {'role': 'user', 'content': f"""Here is a CV:\n{cv_text}\n\nHere is a job description:\n{job_description}\n\nPlease suggest edits or rewrite the CV to better match the job description."""}
]


# call API
response = llm.invoke(messages)

In [ ]:
# print out the results
response.pretty_print()

**Think**

1. If you're building an app that ONLY optimize CV for the FintechX, how will you change your prompts?

2. You can do the same thing in ChatGPT's App. What're the benefits of doing it programmably?